# Train the Module 6 Conformer on Google Colab (free tier)

**Why the Drive mount below is not optional.** Free Colab disconnects on idle
(~90 min) and caps sessions (~12 h), and a disconnect destroys the VM's disk. A
3-hour training run *will* be interrupted. That is survivable only because
`scripts/11_train_asr_conformer.py` checkpoints model **+ optimizer + step**
(`src/checkpoint.py`) — so if `outputs/` lives on Drive, you reconnect, re-run
the same cell, and it resumes from the last eval interval. If `outputs/` lives
on the VM, every disconnect costs you the whole run.

**Honest comparison.** Free T4 ≈ an RTX A4000, so this is roughly the same speed
as the rented option and costs $0 instead of ~$0.50. What you pay instead is
babysitting: no SSH, no automation, and reconnecting by hand. Use Colab when you
want it free and are around; use RunPod when you want it unattended.

> Run the local validation gate **before** this notebook:
> `uv run pytest -q && uv run python scripts/14_validate.py`

## 1. Confirm a GPU was actually allocated
Free Colab does not guarantee one. If this says 'no GPU', change
*Runtime → Change runtime type → T4 GPU*, or come back later.

In [ ]:
!nvidia-smi || echo 'NO GPU — set Runtime > Change runtime type > T4 GPU'

## 2. Mount Drive — this is what makes disconnects survivable

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = '/content/drive/MyDrive/vad-asr-outputs'
os.makedirs(CKPT_DIR, exist_ok=True)
print('checkpoints will persist in', CKPT_DIR)

## 3. Get the code
Either push this repo somewhere and set `REPO_URL`, or zip it locally
(`git archive -o repo.zip HEAD`) and upload it with the file browser.

In [ ]:
REPO_URL = ''  # e.g. 'https://github.com/<you>/VAD-ASR.git'

import os, pathlib
if REPO_URL:
    !git clone -q $REPO_URL /content/project
elif pathlib.Path('/content/repo.zip').exists():
    !mkdir -p /content/project && unzip -q -o /content/repo.zip -d /content/project
else:
    raise SystemExit('set REPO_URL, or upload repo.zip to /content/')
os.chdir('/content/project')
print(os.getcwd(), os.listdir()[:12])

## 4. Install dependencies
`uv sync` pulls the CUDA build of torch automatically on Linux — nothing in
`pyproject.toml` needs changing for GPU.

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh -s -- -q
import os
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']
!uv sync -q
!uv run python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## 5. Point `outputs/` at Drive
A symlink, so the training script needs no changes and every checkpoint it
writes lands on Drive automatically.

In [ ]:
import os, shutil, pathlib
p = pathlib.Path('outputs')
if p.is_symlink():
    p.unlink()
elif p.exists():
    shutil.rmtree(p)
os.symlink(CKPT_DIR, 'outputs')
print('outputs ->', os.readlink('outputs'))
print('already there:', os.listdir('outputs'))

## 6. Validate locally on the Colab box first
Same gate as on the laptop. Three minutes here beats discovering a bug at
hour three.

In [ ]:
!uv run pytest -q
!SIZE=small VOCAB=1024 uv run python scripts/14_validate.py

## 7. Train
`RESUME=1` is the default. **If you get disconnected, just re-run this cell** —
it picks up from the last eval interval instead of starting over.

`DOWNLOAD=1` fetches LibriSpeech train-clean-100 (~6.3 GB) on first run. Put
`data/` on the VM, not Drive: Drive I/O is slow and re-downloading is faster
than reading 28k small files over it.

In [ ]:
!SPLIT=train-clean-100 DOWNLOAD=1 SIZE=small STEPS=40000 VOCAB=1024 \
  EVAL_EVERY=2000 AUG_START=4000 ACCUM=2 MAX_FRAMES=24000 \
  uv run python scripts/11_train_asr_conformer.py

## 8. Results
Everything is already on Drive. Compare against the laptop baseline
(WER 0.607 offline / 0.627 streaming, trained on 4.1 h) with
`scripts/13_asr_compare.py`.

In [ ]:
!ls -la outputs/
!uv run python scripts/13_asr_compare.py